# Linear Discriminant Analysis (LDA)

## Importing the libraries

In [ ]:
# numpy       -> array handling (used by the plotting cells)
# matplotlib  -> drawing the decision-boundary charts
# pandas      -> loading the dataset
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Importing the dataset

In [ ]:
# read the wine dataset
dataset = pd.read_csv('data/Wine.csv')

# X = every column except the last (the 13 chemical measurements)
# y = the last column (Customer_Segment - the wine type, 1, 2 or 3)
X = dataset.iloc[:, :-1].values
y = dataset.iloc[:, -1].values

## Splitting the dataset into the Training set and Test set

In [ ]:
# split into training (80%) and test (20%) sets
# random_state=0 makes the split reproducible - same rows every run
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

## Feature Scaling

In [ ]:
# standardise the features: rescale each column to mean 0, standard deviation 1
# fit_transform on TRAIN learns the means and stds; transform on TEST reuses them
# (never fit on the test set - that would leak test information into the model)
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

## Applying LDA

In [ ]:
# LDA reduces the 13 measurements down to 2 new features (LD1 and LD2)
# note that fit_transform receives y_train as well - LDA is SUPERVISED:
# it uses the wine types to choose the 2 directions that best separate the classes
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
lda = LDA(n_components = 2)
X_train = lda.fit_transform(X_train, y_train)
X_test = lda.transform(X_test)   # the test set is transformed with the same directions

## Training the Logistic Regression model on the Training set

In [ ]:
# train Logistic Regression on the 2 LDA features
from sklearn.linear_model import LogisticRegression
classifier = LogisticRegression(random_state = 0)
classifier.fit(X_train, y_train)

## Making the Confusion Matrix

In [ ]:
# predict the wine type for the test set and count the results
# rows = actual type, columns = predicted type
from sklearn.metrics import confusion_matrix, accuracy_score
y_pred = classifier.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print(cm)
accuracy_score(y_test, y_pred)

## Visualising the Training set results

In [ ]:
# draw the decision regions: colour every point of the 2-D plane by the
# class the model would predict there, then plot the real wines on top
from matplotlib.colors import ListedColormap
X_set, y_set = X_train, y_train

# a fine grid of points covering the whole plot area
X1, X2 = np.meshgrid(np.arange(start = X_set[:, 0].min() - 1, stop = X_set[:, 0].max() + 1, step = 0.01),
                     np.arange(start = X_set[:, 1].min() - 1, stop = X_set[:, 1].max() + 1, step = 0.01))

# predict a class for every grid point and paint the three regions
plt.contourf(X1, X2, classifier.predict(np.array([X1.ravel(), X2.ravel()]).T).reshape(X1.shape),
             alpha = 0.75, cmap = ListedColormap(('red', 'green', 'blue')))
plt.xlim(X1.min(), X1.max())
plt.ylim(X2.min(), X2.max())

# plot the actual training wines, one colour per true class
for i, j in enumerate(np.unique(y_set)):
    plt.scatter(X_set[y_set == j, 0], X_set[y_set == j, 1],
                color = ['red', 'green', 'blue'][i], label = j)
plt.title('Logistic Regression (Training set)')
plt.xlabel('LD1')
plt.ylabel('LD2')
plt.legend()
plt.show()

## Visualising the Test set results

In [ ]:
# same picture as above, but with the TEST wines - data the model never trained on
from matplotlib.colors import ListedColormap
X_set, y_set = X_test, y_test

X1, X2 = np.meshgrid(np.arange(start = X_set[:, 0].min() - 1, stop = X_set[:, 0].max() + 1, step = 0.01),
                     np.arange(start = X_set[:, 1].min() - 1, stop = X_set[:, 1].max() + 1, step = 0.01))

plt.contourf(X1, X2, classifier.predict(np.array([X1.ravel(), X2.ravel()]).T).reshape(X1.shape),
             alpha = 0.75, cmap = ListedColormap(('red', 'green', 'blue')))
plt.xlim(X1.min(), X1.max())
plt.ylim(X2.min(), X2.max())

for i, j in enumerate(np.unique(y_set)):
    plt.scatter(X_set[y_set == j, 0], X_set[y_set == j, 1],
                color = ['red', 'green', 'blue'][i], label = j)
plt.title('Logistic Regression (Test set)')
plt.xlabel('LD1')
plt.ylabel('LD2')
plt.legend()
plt.show()

<h1>What did we learn?</h1>

- **LDA is supervised dimensionality reduction.** Unlike PCA in the previous lab,
  LDA is given the class labels and uses them to choose the 2 directions that best
  separate the three wine types.
- **13 measurements became 2 features (LD1 and LD2)** — and because those directions
  were chosen to separate the classes, the three wine types form three clean,
  well-separated groups on the plot.
- **A simple model was enough after LDA.** Logistic Regression on just the 2 LDA
  features classified the test set perfectly — the confusion matrix has every wine
  on the diagonal.
- **The decision-boundary plots show how the model thinks.** The coloured regions
  are the model's prediction for every possible point; the dots are real wines.
  Where a dot sits in its own colour, the model got it right.
- **Same rule as always:** the scaler and LDA were fitted on the training set only,
  then applied to the test set — the test data stayed unseen until marking time.